# Titanic · устойчивость моделей к разбиению

Вопрос: сохраняется ли результат выбранных моделей при смене состава фолдов? Проверяем Dummy и семь моделей на seeds разбиения 42, 17 и 123, по пять стратифицированных фолдов. Seed моделей — 42; параметры и признаки фиксированы, нового подбора нет.

**Посмотреть результаты:** ячейка импортов → раздел «Сравнение по разбиениям». Он читает сохранённый CSV и не обучает модели.

**Повторить исследование:** выполнить все ячейки сверху вниз — 120 обучений, файл `experiments/model_seed_stability.csv` заменяется. Сетки из [modeling.ipynb](modeling.ipynb) не нужны.

Средняя Accuracy — основная метрика, ROC-AUC — дополнительная. Три разбиения используют один набор пассажиров; это проверка устойчивости ранее выбранных решений, а не независимый тест.


In [ ]:
import os
import sys
from pathlib import Path

# Ядро может стартовать из корня проекта или из notebooks.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config.yaml").is_file() or not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Откройте ноутбук из корня Titanic-ML или папки notebooks")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # Пути в конфиге и ячейках заданы от корня проекта.

import json
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from src.sklearn_experiments import build_experiment


df_train = pd.read_csv("data/train.csv")


## 1. Фиксированные конфигурации

Сохраняем выбранные в modeling параметры, признаки и обработку. KNN и LR используют масштабирование, дерево и RF — общий pipeline без него, XGBoost — one-hot, бустинги ниже — встроенные категории.


In [ ]:
logreg_best_config = {
    "features": {
        "Age": "numeric",
        "Sex": "binary",
        "Embarked": "onehot",
        "Pclass": "onehot",
        "_Fare": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },

    "fare_log1p": True,
    "sex_pclass": True,
    "age_squared": True,
    "is_alone": True,

    "is_child": False,
    "family_size": False,
    "scale_numeric": True,
}

knn_best_config = {
    "features": {
        "Age": "numeric",
        "_Fare": "numeric",
        "Sex": "onehot",
        "Embarked": "onehot",
        "Pclass": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },
    "scale_numeric": True,
    "family_size": True,
    "is_alone": False,
    "is_child": False,
    "fare_log1p": False,
    "age_squared": False,
    "sex_pclass": False,
}

tree_best_config = {
    "features": {
        "Age": "numeric",
        "Fare": "numeric",
        "Sex": "binary",
        "Embarked": "onehot",
        "Pclass": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },
    "scale_numeric": False,
    "sex_pclass": True,
}

rf_config = {
    "features": {
        "Age": "numeric",
        "Fare": "numeric",
        "Sex": "binary",
        "Embarked": "onehot",
        "Pclass": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },
    "scale_numeric": False,
}

stability_models = {
    "Dummy": (DummyClassifier(strategy="most_frequent"), rf_config),
    "Logistic Regression": (
        LogisticRegression(solver="saga", C=1, l1_ratio=0.0, max_iter=1000, random_state=42),
        logreg_best_config,
    ),
    "KNN": (KNeighborsClassifier(n_neighbors=21, weights="uniform", metric="manhattan"), knn_best_config),
    "Decision Tree": (
        DecisionTreeClassifier(max_depth=8, min_samples_split=10, min_samples_leaf=1, criterion="gini", random_state=42),
        tree_best_config,
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, max_depth=None, max_features=None, min_samples_leaf=4, min_samples_split=10, random_state=42),
        rf_config,
    ),
    "XGBoost": (
        XGBClassifier(n_estimators=80, learning_rate=0.1, max_depth=4, min_child_weight=5, random_state=42, eval_metric="logloss", n_jobs=1),
        rf_config,
    ),
}



## 2. Данные и оценочные модели

Производные признаки вычисляются построчно. Медиана, scaler и one-hot обучаются внутри train-фолда. CatBoost получает категории напрямую, числовые пропуски обрабатывает сам; LightGBM получает pandas category.

Для CatBoost + Title фиксированы 110 деревьев. Здесь обычный fit на общих sklearn-фолдах; исторические оценки catboost.cv относятся к другой процедуре и сохраняются отдельно.


In [ ]:
stability_candidates = {}
for name, (model, feature_config) in stability_models.items():
    model_X, model_y, pipeline = build_experiment(df_train, "Survived", model, feature_config)
    stability_candidates[name] = (model_X, pipeline, feature_config, model.get_params())

native_features = ["Age", "Fare", "Sex", "Embarked", "Pclass", "SibSp", "Parch"]
native_categories = ["Sex", "Embarked", "Pclass"]
catboost_X = df_train[native_features].copy()
for column in native_categories:
    catboost_X[column] = catboost_X[column].fillna("Missing").astype(str)
titles = df_train["Name"].str.extract(r",\s*([^.]*)\.", expand=False).fillna("Other")
catboost_X["Title"] = titles.where(titles.isin(["Mr", "Mrs", "Miss", "Master"]), "Other")
catboost_fixed = CatBoostClassifier(
    iterations=110, depth=7, learning_rate=0.1, l2_leaf_reg=3, random_strength=0,
    loss_function="Logloss", random_seed=42, verbose=False, allow_writing_files=False,
    thread_count=1,
)
catboost_fixed_config = {
    "features": list(catboost_X.columns), "cat_features": native_categories + ["Title"],
    "categorical_nan": "Missing", "numeric_nan": "native",
}
stability_candidates["CatBoost + Title"] = (
    catboost_X, catboost_fixed, catboost_fixed_config, catboost_fixed.get_params(),
)

lightgbm_X = df_train[native_features].copy()
for column in native_categories:
    lightgbm_X[column] = lightgbm_X[column].astype("category")
lightgbm_fixed = LGBMClassifier(
    n_estimators=30, learning_rate=0.2, num_leaves=31, min_child_samples=40,
    random_state=42, verbosity=-1, n_jobs=1,
)
stability_candidates["LightGBM native"] = (
    lightgbm_X, lightgbm_fixed,
    {"features": native_features, "cat_features": native_categories, "numeric_nan": "native"},
    lightgbm_fixed.get_params(),
)



## 3. Повторный запуск — 120 обучений

Внешний цикл меняет только разбиение, внутренний сравнивает все модели на одинаковых фолдах. По каждой оценке сохраняем seed, фолд, признаки, параметры, Accuracy и ROC-AUC. Для просмотра готовых результатов эту ячейку пропустите.


In [ ]:
stability_y = df_train["Survived"]
stability_rows = []
for split_seed in [42, 17, 123]:
    common_splits = list(StratifiedKFold(
        n_splits=5, shuffle=True, random_state=split_seed
    ).split(df_train, stability_y))
    for name, (model_X, estimator, feature_config, params) in stability_candidates.items():
        assert model_X.index.equals(stability_y.index)
        scores = cross_validate(
            estimator, model_X, stability_y, cv=common_splits,
            scoring=["accuracy", "roc_auc"], error_score="raise", n_jobs=1,
            params={"cat_features": native_categories + ["Title"]} if name == "CatBoost + Title" else None,
        )
        for fold, (accuracy, roc_auc) in enumerate(zip(
            scores["test_accuracy"], scores["test_roc_auc"]
        ), start=1):
            stability_rows.append({
                "model": name, "split_seed": split_seed, "model_seed": 42 if name not in ["Dummy", "KNN"] else None,
                "fold": fold, "accuracy": accuracy, "roc_auc": roc_auc,
                "features": json.dumps(list(model_X.columns)),
                "config": json.dumps(feature_config),
                "model_params": json.dumps(params, default=str),
            })
    print(f"Seed разбиения {split_seed}: проверены {len(stability_candidates)} моделей")

model_stability_folds = pd.DataFrame(stability_rows)
model_stability_path = Path("experiments/model_seed_stability.csv")
model_stability_path.parent.mkdir(parents=True, exist_ok=True)
model_stability_folds.to_csv(model_stability_path, index=False)



## 4. Сравнение по разбиениям

Колонки 42, 17 и 123 — средняя Accuracy пяти фолдов соответствующего seed. `accuracy_mean` — среднее трёх средних; `accuracy_split_std` — стандартное отклонение между разбиениями (ddof=0), не между фолдами. `roc_auc_mean` — средний ROC-AUC по тем же оценкам.


In [ ]:
model_stability_folds = pd.read_csv("experiments/model_seed_stability.csv")

model_stability_seeds = model_stability_folds.groupby(
    ["model", "split_seed"], sort=False
).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_fold_std=("accuracy", lambda values: values.std(ddof=0)),
    roc_auc_mean=("roc_auc", "mean"),
).reset_index()
model_stability_accuracy = model_stability_seeds.pivot(
    index="model", columns="split_seed", values="accuracy_mean"
)[[42, 17, 123]]
model_stability_summary = model_stability_seeds.groupby("model", sort=False).agg(
    accuracy_mean=("accuracy_mean", "mean"),
    accuracy_split_std=("accuracy_mean", lambda values: values.std(ddof=0)),
    roc_auc_mean=("roc_auc_mean", "mean"),
).sort_values("accuracy_mean", ascending=False)
model_stability_accuracy = model_stability_accuracy.reindex(model_stability_summary.index)
model_stability_report = model_stability_accuracy.join(model_stability_summary)
model_stability_report.round(5)


,42,17,123,accuracy_mean,accuracy_split_std,roc_auc_mean
model,,,,,,
XGBoost,0.84287,0.83274,0.83280,0.83613,0.00476,0.87095
Random Forest,0.84621,0.82601,0.83393,0.83538,0.00831,0.87214
LightGBM native,0.84287,0.82936,0.83055,0.83426,0.00611,0.86899
CatBoost + Title,0.82601,0.83838,0.83054,0.83164,0.00511,0.86748
Decision Tree,0.84287,0.81031,0.82382,0.82566,0.01336,0.83553
KNN,0.83166,0.82152,0.81596,0.82305,0.00650,0.85467
Logistic Regression,0.81594,0.81592,0.81148,0.81445,0.00210,0.86193
Dummy,0.61616,0.61616,0.61616,0.61616,0.00000,0.50000


## Вывод по разным разбиениям

XGBoost, Random Forest и LightGBM получили близкие средние Accuracy: 83.61%, 83.54% и 83.43%. XGBoost меньше менялся между этими разбиениями, но преимущество 0.08 п.п. над RF не доказывает превосходства. RF имеет несколько больший средний ROC-AUC (0.87214 против 0.87095). Дерево сильнее зависит от разбиения (std 1.34 п.п.), у LR разброс меньше (0.21 п.п.), но Accuracy ниже.

CatBoost + Title получил 82.60%, 83.84% и 83.05% на общих фолдах. Это новый сопоставимый запуск с фиксированными 110 деревьями; прежние значения собственного catboost.cv сохраняем как отдельное исследование.

Нового подбора не проводили. Проверка не даёт основания для уверенного выбора единственного победителя среди трёх лидеров; практический кандидат RF остаётся обоснованным. Повторно подбирать ансамбли из новых лидеров сейчас не требуется. Числа доступны в `experiments/model_seed_stability.csv`.
